# Driver Drowsiness Detection — Week 3
### CNN Architecture Design + EAR/MAR Module Implementation
NTCC | Amity School of Engineering & Technology | May 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import distance as dist
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense
)
from tensorflow.keras.optimizers import Adam

DRIVE  = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
RES    = f'{DRIVE}/results'
MODELS = f'{DRIVE}/models'
os.makedirs(RES,    exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

IMG_SIZE    = 64
NUM_CLASSES = 4

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Loading preprocessed data from Week 2

In [ ]:
X_train = np.load(f'{DRIVE}/data/X_train.npy')
X_val   = np.load(f'{DRIVE}/data/X_val.npy')
X_test  = np.load(f'{DRIVE}/data/X_test.npy')
y_train = np.load(f'{DRIVE}/data/y_train.npy')
y_val   = np.load(f'{DRIVE}/data/y_val.npy')
y_test  = np.load(f'{DRIVE}/data/y_test.npy')

print('X_train:', X_train.shape)
print('X_val  :', X_val.shape)
print('X_test :', X_test.shape)
print('pixel range:', X_train.min(), '-', X_train.max())

## CNN Architecture

Going with 3 conv blocks. Filters increase as we go deeper (32 → 64 → 128).
Added BatchNorm after each conv block to stabilise training.
Dropout at 0.25 in conv blocks and 0.5 before final dense layer to prevent overfitting.

In [ ]:
model = Sequential([
    # block 1
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.50),
    Dense(NUM_CLASSES, activation='softmax')
], name='Drowsiness_CNN')

model.compile(
    optimizer = Adam(learning_rate=0.0005),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

model.summary()

In [ ]:
# quick sanity check — forward pass on 8 samples
test_out = model.predict(X_train[:8], verbose=0)
print('Input shape  :', X_train[:8].shape)
print('Output shape :', test_out.shape)
print('Sample output:', test_out[0].round(3))
print('Sums to 1    :', test_out[0].sum().round(4))
print('Test forward pass OK')

## Feature map visualisation

Checking what each conv layer actually picks up from an eye image.

In [ ]:
conv_layers  = [l for l in model.layers if 'conv' in l.name]
layer_outputs = [l.output for l in conv_layers]
vis_model     = Model(inputs=model.input, outputs=layer_outputs)

fmaps = vis_model.predict(X_train[0:1], verbose=0)

fig, axes = plt.subplots(len(conv_layers), 4, figsize=(13, 3.5 * len(conv_layers)))
fig.suptitle('Feature maps — what each Conv layer detects', fontsize=12)

for row, (name, fmap) in enumerate(zip([l.name for l in conv_layers], fmaps)):
    for col in range(4):
        axes[row][col].imshow(fmap[0, :, :, col], cmap='viridis')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(name, fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RES}/feature_maps.png', dpi=130)
plt.show()

## EAR — Eye Aspect Ratio

Formula: EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)

When eye is open → EAR around 0.25–0.35
When eye is closing → EAR drops below 0.20

In [ ]:
def calculate_EAR(eye_pts):
    """
    6 landmark points around the eye.
    Returns Eye Aspect Ratio.
    """
    A = dist.euclidean(eye_pts[1], eye_pts[5])
    B = dist.euclidean(eye_pts[2], eye_pts[4])
    C = dist.euclidean(eye_pts[0], eye_pts[3])
    return round((A + B) / (2.0 * C), 4)


# testing with dummy coordinates
open_eye   = [(0,0),(1,2),(2,2),(4,0),(3,2),(1,2)]
closed_eye = [(0,0),(1,0.2),(2,0.2),(4,0),(3,0.2),(1,0.2)]

print('EAR open eye  :', calculate_EAR(open_eye))
print('EAR closed eye:', calculate_EAR(closed_eye))
print('Threshold: 0.25 — below this the eye is considered closing')

## MAR — Mouth Aspect Ratio

Same idea as EAR but applied to 8 mouth landmark points.
Normal closed mouth → MAR around 0.2–0.3
Yawning → MAR goes above 0.5

In [ ]:
def calculate_MAR(mouth_pts):
    """
    8 landmark points around the mouth.
    Returns Mouth Aspect Ratio.
    """
    A = dist.euclidean(mouth_pts[1], mouth_pts[7])
    B = dist.euclidean(mouth_pts[2], mouth_pts[6])
    C = dist.euclidean(mouth_pts[3], mouth_pts[5])
    D = dist.euclidean(mouth_pts[0], mouth_pts[4])
    return round((A + B + C) / (3.0 * D), 4)


closed_mouth = [(0,0),(1,0.3),(2,0.3),(3,0.3),(6,0),(5,0.3),(4,0.3),(3,0.3)]
open_mouth   = [(0,0),(1,1.5),(2,1.5),(3,1.5),(6,0),(5,1.5),(4,1.5),(3,1.5)]

print('MAR closed mouth:', calculate_MAR(closed_mouth))
print('MAR open mouth  :', calculate_MAR(open_mouth))
print('Threshold: 0.50 — above this the person is yawning')

## Drowsiness score — combining all three signals

Combining EAR, MAR and head tilt into one score between 0 and 1.
Weights: 40% EAR + 30% MAR + 30% head tilt.

In [ ]:
def drowsiness_score(ear, mar, head_tilt,
                      ear_thresh=0.25, mar_thresh=0.50, tilt_thresh=20):
    """
    Weighted fusion of three drowsiness signals.
    Returns a score between 0.0 (awake) and 1.0 (drowsy).
    """
    ear_sig  = max(0, (ear_thresh - ear)  / ear_thresh)
    mar_sig  = max(0, (mar - mar_thresh)  / (1 - mar_thresh))
    tilt_sig = max(0, (head_tilt - tilt_thresh) / (90 - tilt_thresh))

    score = (0.40 * ear_sig) + (0.30 * mar_sig) + (0.30 * tilt_sig)
    return round(min(score, 1.0), 4)


print('Test cases:')
print('  Fully awake   :', drowsiness_score(0.30, 0.25, 5))
print('  Eyes closing  :', drowsiness_score(0.18, 0.25, 5))
print('  Yawning       :', drowsiness_score(0.28, 0.70, 5))
print('  Head tilting  :', drowsiness_score(0.28, 0.25, 35))
print('  Fully drowsy  :', drowsiness_score(0.10, 0.80, 50))

In [ ]:
# EAR/MAR threshold visualisation — simulating what the real-time system will track
np.random.seed(7)
frames = np.arange(100)

ear_vals = np.concatenate([
    np.random.uniform(0.28, 0.35, 40),
    np.random.uniform(0.10, 0.20, 30),
    np.random.uniform(0.28, 0.35, 30)
])
mar_vals = np.concatenate([
    np.random.uniform(0.20, 0.32, 50),
    np.random.uniform(0.55, 0.78, 20),
    np.random.uniform(0.20, 0.32, 30)
])

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
fig.suptitle('EAR and MAR threshold simulation', fontsize=12)

axes[0].plot(frames, ear_vals, color='#1E88E5', lw=1.5, label='EAR')
axes[0].axhline(0.25, color='red', lw=1.5, linestyle='--', label='Threshold (0.25)')
axes[0].fill_between(frames, ear_vals, 0.25,
                      where=(ear_vals < 0.25), alpha=0.25, color='red')
axes[0].set_title('Eye Aspect Ratio over time')
axes[0].set_ylabel('EAR')
axes[0].legend()
axes[0].set_ylim(0, 0.45)

axes[1].plot(frames, mar_vals, color='#FF8F00', lw=1.5, label='MAR')
axes[1].axhline(0.50, color='red', lw=1.5, linestyle='--', label='Threshold (0.50)')
axes[1].fill_between(frames, mar_vals, 0.50,
                      where=(mar_vals > 0.50), alpha=0.25, color='red')
axes[1].set_title('Mouth Aspect Ratio over time')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('MAR')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{RES}/ear_mar_thresholds.png', dpi=150)
plt.show()

In [ ]:
# save model architecture
import json
with open(f'{MODELS}/cnn_architecture.json', 'w') as f:
    json.dump(json.loads(model.to_json()), f, indent=2)
print('Architecture saved')
print(f'Total params: {model.count_params():,}')

In [ ]:
import shutil
from datetime import datetime

REPO = '/content/Driver-Drowsiness-Detection-Using-Deep-Learning-Techniques'
NOTEBOOK = 'Week3_ModelDesign.ipynb'

shutil.copy(f'/content/{NOTEBOOK}', f'{REPO}/notebooks/{NOTEBOOK}')

for fname in ['feature_maps.png', 'ear_mar_thresholds.png']:
    src = f'{RES}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{REPO}/results/{fname}')

os.chdir(REPO)
os.system('git add .')
os.system(f'git commit -m "Week 3: CNN architecture + EAR/MAR modules — {datetime.now().strftime("%d %b %Y")}'+'"')
print(os.popen('git push 2>&1').read())